In [1]:
# Import Packages
import os
import datetime
import zipfile
import requests
import io
import pandas as pd
import re

### Data Ingestion Setup and Preprocessing Utilities

This section defines the target years (2021–2024) and dynamically generates IPEDS download URLs and table names, accounting for differences in data access formats across years. Utility functions are included to standardize ZIP filenames and clean column names for consistent downstream processing.

To add new tables, you will need to stay within the format as below. The table name will need to use the yr variable correctly, and the name will need to be added in all four lists. This is just one way to handle this in an automated way. Hard coding or simply visiting the IPEDS data center to download these tables are also just as proficient.

In [2]:
# Initialize variables and functions
years = [2021, 2022, 2023, 2024]

def generate_table_names(yr):
    if yr < 2023:
        return [
            f'https://nces.ed.gov/ipeds/datacenter/data/C{yr}_B.zip',
            f'https://nces.ed.gov/ipeds/datacenter/data/HD{yr}.zip',
            f'https://nces.ed.gov/ipeds/datacenter/data/F{(yr-1)%100:02}{yr%100:02}_F1A.zip'
        ], [
        f'C{yr}_B',
        f'HD{yr}',
        f'F{(yr-1)%100:02}{yr%100:02}_F1A'
        ]
    else:
        return [
            f'https://nces.ed.gov/ipeds/data-generator?tableName=C{yr}_B&year={yr}',
            f'https://nces.ed.gov/ipeds/data-generator?tableName=HD{yr}&year={yr}',
            f'https://nces.ed.gov/ipeds/data-generator?tableName=F{(yr-1)%100:02}{yr%100:02}_F1A&year={yr}'
        ], [
        f'C{yr}_B',
        f'HD{yr}',
        f'F{(yr-1)%100:02}{yr%100:02}_F1A'
        ]
def lowercase_zip_filenames(z):
    """Takes a ZipFile object `z` and returns a new ZipFile object with all filenames converted to lowercase."""
    new_zip_bytes = io.BytesIO()

    with zipfile.ZipFile(new_zip_bytes, 'w') as new_zip:
        for item in z.infolist():
            data = z.read(item.filename)
            new_info = zipfile.ZipInfo(item.filename.lower())
            new_info.date_time = item.date_time
            new_info.compress_type = item.compress_type
            new_zip.writestr(new_info, data)

    new_zip_bytes.seek(0)
    z_lower = zipfile.ZipFile(new_zip_bytes, 'r')
    return z_lower

def strip_spaces(string_list):
    """Strips spaces, replaces periods with underscores, and removes non-alphanumeric or underscore characters."""
    return [
        re.sub(r'[^A-Za-z0-9_]', '', s.strip().replace('.', '_')) if isinstance(s, str) else s
        for s in string_list
    ]

### Data Retrieval by Year

IPEDS data access differs by year: datasets prior to 2023 are available as direct ZIP downloads, while 2023 and later data must be retrieved through the IPEDS data generator interface. The logic above dynamically selects the appropriate URL format to ensure consistent access across years.

For each dataset, the code checks for the presence of revised (“_RV”) files, which represent updated or corrected versions, and prioritizes these when available. Files are then standardized and saved locally in a structured directory (`data/{year}/`) to support reproducibility and downstream processing.

In [3]:
### URL to use prior to 2023
# ipeds_locs_2022 = 'https://nces.ed.gov/ipeds/datacenter/data/'
### URL for 2023 and after
# ipeds_locs_2023 = 'https://nces.ed.gov/ipeds/data-generator?tableName=C2024_B&year=2024'


for yr in years:
    ipeds_url_list, ipeds_fils_list = generate_table_names(yr)

    for i in range(len(ipeds_url_list)):
        try:
            print(f'[{yr}] GETTING FILES FROM {ipeds_fils_list[i]}')
            print(ipeds_fils_list[i])
            rdata = requests.get(ipeds_url_list[i])
            rdata_zip = zipfile.ZipFile(io.BytesIO(rdata.content))
            rdata_zip = lowercase_zip_filenames(rdata_zip)

            # Determine correct filename
            if ipeds_fils_list[i].lower() + '_rv.csv' in rdata_zip.namelist():
                print(f'{ipeds_fils_list[i]} has final revision.')
                file_key = ipeds_fils_list[i].lower() + '_rv.csv'
                table_name = ipeds_fils_list[i] + '_RV'
            else:
                file_key = ipeds_fils_list[i].lower() + '.csv'
                table_name = ipeds_fils_list[i]

            with rdata_zip.open(file_key) as file:
                df = pd.read_csv(file, encoding='ISO-8859-1')

                df.columns = strip_spaces(df.columns)

                output_path = f'data/{yr}/{table_name}.csv'
                os.makedirs(os.path.dirname(output_path), exist_ok=True)

                print(f'Saving {table_name} to {output_path}')
                df.to_csv(output_path, index=False)

        except zipfile.BadZipFile as e:
            print(f'{ipeds_fils_list[i]} File not in IPEDS Data Center: {type(e).__name__}\n')
            continue


[2021] GETTING FILES FROM C2021_B
C2021_B
C2021_B has final revision.
Saving C2021_B_RV to data/2021/C2021_B_RV.csv
[2021] GETTING FILES FROM HD2021
HD2021
Saving HD2021 to data/2021/HD2021.csv
[2021] GETTING FILES FROM F2021_F1A
F2021_F1A
F2021_F1A has final revision.
Saving F2021_F1A_RV to data/2021/F2021_F1A_RV.csv
[2022] GETTING FILES FROM C2022_B
C2022_B
C2022_B has final revision.
Saving C2022_B_RV to data/2022/C2022_B_RV.csv
[2022] GETTING FILES FROM HD2022
HD2022
Saving HD2022 to data/2022/HD2022.csv
[2022] GETTING FILES FROM F2122_F1A
F2122_F1A
F2122_F1A has final revision.
Saving F2122_F1A_RV to data/2022/F2122_F1A_RV.csv
[2023] GETTING FILES FROM C2023_B
C2023_B
C2023_B has final revision.
Saving C2023_B_RV to data/2023/C2023_B_RV.csv
[2023] GETTING FILES FROM HD2023
HD2023
Saving HD2023 to data/2023/HD2023.csv
[2023] GETTING FILES FROM F2223_F1A
F2223_F1A
F2223_F1A has final revision.
Saving F2223_F1A_RV to data/2023/F2223_F1A_RV.csv
[2024] GETTING FILES FROM C2024_B
C2024_

### Cross-Year File Union

This step constructs unified datasets across multiple reporting years by combining annual CSV extracts for each IPEDS table. For each table-year combination, the function checks for both standard and revised (“_RV”) versions of the file, prioritizing revised data when available.

A `YEAR` field is appended to preserve temporal context, and a `REVISED` flag is added to indicate whether the observation originates from a revised dataset. All available yearly files are then concatenated into a single DataFrame, with automatic column alignment handled during the union process.

In [4]:
# Initializing for collection unions
dataset_names = {
'cyear_b':             [f'C{yr}_B' for yr in years],
'fyryr_f1a':           [f'F{(yr-1)%100:02}{(yr)%100:02}_F1A' for yr in years],
}
def union_csv(table_list, years):
    dfs = []

    for table_name, yr in zip(table_list, years):
        

        if os.path.exists(f'data/{yr}/{table_name}.csv'):
            df = pd.read_csv(f'data/{yr}/{table_name}.csv')
            df['YEAR'] = yr
            df['REVISED'] = 'N'

            dfs.append(df)
        elif os.path.exists(f'data/{yr}/{table_name}_RV.csv'):
            df = pd.read_csv(f'data/{yr}/{table_name}_RV.csv')
            df['YEAR'] = yr  
            df['REVISED'] = 'Y'

            dfs.append(df)
        else:
            print(f'Missing: {table_name}')
            
            continue

    if not dfs:
        return pd.DataFrame()

    # pandas will auto-align columns during concat
    return pd.concat(dfs, ignore_index=True)

The resulting unified tables are then written to a consolidated directory (`data/unioned_table/`) to support downstream analysis across all years in a consistent format.

In [5]:
os.makedirs('data/unioned_table', exist_ok=True)

for key in dataset_names:
    print(f'Processing {key}')

    unioned_df = union_csv(dataset_names[key], years)

    if unioned_df.empty:
        print(f'No data for {key}')
        continue

    output_path = f'data/unioned_table/{key.upper()}.csv'
    unioned_df.to_csv(output_path, index=False)

    print(f'Saved → {output_path}')

Processing cyear_b
Saved → data/unioned_table/CYEAR_B.csv
Processing fyryr_f1a
Saved → data/unioned_table/FYRYR_F1A.csv


### Data Integration and Filtering

This step integrates the previously unioned datasets by merging institutional cost data (`CYEAR_B`) with financial data (`FYRYR_F1A`) using `UNITID` and `YEAR` as join keys. Institutional characteristics from `HD2024` are then joined to provide contextual information, including institution name.

Column names are standardized to ensure consistency across datasets prior to merging. The combined dataset is then filtered to retain only observations corresponding to Wichita State University–Campus of Applied Sciences and Technology, establishing a single-institution panel for subsequent analysis.

In [6]:
# Load unioned tables
cyear_b = pd.read_csv('data/unioned_table/CYEAR_B.csv')
fyryr_f1a = pd.read_csv('data/unioned_table/FYRYR_F1A.csv')

# Load HD2024 for filtering
hd2024 = pd.read_csv('data/2024/HD2024.csv')

# Standardize column names just in case
cyear_b.columns = cyear_b.columns.str.upper()
fyryr_f1a.columns = fyryr_f1a.columns.str.upper()
hd2024.columns = hd2024.columns.str.upper()

# Join CYEAR_B with FYRYR_F1A on UNITID + YEAR
df = pd.merge(
    cyear_b,
    fyryr_f1a[['UNITID', 'YEAR', 'F1C191']],
    on=['UNITID', 'YEAR'],
    how='inner'
)

# Join with HD2024 to get INSTNM
df = pd.merge(
    df,
    hd2024[['UNITID', 'INSTNM']],
    on='UNITID',
    how='inner'
)

# Filter to Wichita State Tech
df = df[
    df['INSTNM'] == 'Wichita State University-Campus of Applied Sciences and Technology'
]


### Outcome Metric Construction and Export

The Outcome Efficiency Index is calculated as a normalized performance metric by scaling total cost (`CSTOTLT`) relative to completions (`F1C191`). This provides a standardized measure of cost efficiency across years for the selected institution.

The dataset is then reduced to key analytical variables: institutional identifier, institution name, year, input cost, output measure, and the computed efficiency index.

Finally, the resulting panel dataset is exported as a CSV file for downstream analysis and reporting.

In [7]:
# Calculate Outcome Efficiency Index
df['OUTCOME_EFFICIENCY_INDEX'] = df['CSTOTLT'] * 100000 / df['F1C191']

# Select final columns
final_df = df[['UNITID', 'INSTNM', 'YEAR', 'CSTOTLT', 'F1C191', 'OUTCOME_EFFICIENCY_INDEX']]

# Save result
output_path = 'data/metrics/Wichita_State_Efficiency.csv'
final_df.to_csv(output_path, index=False)

print(f'Saved → {output_path}')

Saved → data/metrics/Wichita_State_Efficiency.csv
